In [ ]:
!pip install -qU streamlit pyngrok \
langchain langchain-community langchain-openai langchain-classic \
faiss-cpu unstructured tiktoken openai

In [ ]:
!npm install -g localtunnel

In [1]:
print("Hello word")

Hello word


In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# NEW LCEL tools
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
from google.colab import userdata
userdata.get('open_api_key')

In [25]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("open_api_key")

In [26]:
%%writefile app.py
import os
import streamlit as st
import pickle

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# =========================
# READ ENV VARIABLE ONLY
# =========================
api_key = os.getenv("OPENAI_API_KEY")

st.title("📈 RockyBot: News Research Tool")

urls = [st.text_input(f"URL {i+1}") for i in range(3)]
process = st.button("Process URLs")

file_path = "faiss_store.pkl"

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    api_key=api_key
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key
)

if process:
    loader = UnstructuredURLLoader(urls=urls)
    data = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    docs = splitter.split_documents(data)

    vectorstore = FAISS.from_documents(docs, embeddings)

    with open(file_path, "wb") as f:
        pickle.dump(vectorstore, f)

    st.success("Done processing!")

if os.path.exists(file_path):
    with open(file_path, "rb") as f:
        vectorstore = pickle.load(f)
    retriever = vectorstore.as_retriever()
else:
    retriever = None

prompt = ChatPromptTemplate.from_template("""
Answer ONLY from context:

{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

if retriever:
    chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )
else:
    chain = None

query = st.text_input("Ask a question:")

if query and chain:
    st.write(chain.invoke(query))

Overwriting app.py


In [27]:
!streamlit run app.py &>/content/logs.txt &

In [21]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧⠇your url is: https://sweet-lions-doubt.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.115.179.22:8501

  Stopping...
^C


In [ ]:
!lt --port 8501

your url is: https://ready-trains-hang.loca.lt
